In [28]:
import torch
from datasets import load_from_disk
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

In [23]:
# 1. Carica il dataset tokenizzato di Geneformer
dataset_path = (
    "/projects/shared/intronic_bam/datasets/geneformer/be1.dataset"
)
ds = load_from_disk(dataset_path)

In [24]:
ds

Dataset({
    features: ['input_ids', 'Sample', 'Barcode', 'length'],
    num_rows: 29128
})

In [29]:
# 2. Definisci una collate_fn personalizzata usando il pad token di Geneformer (0)
def geneformer_collate_fn(batch):
    # Converte la lista di token di ogni cellula in un tensore PyTorch
    input_ids_list = [
        torch.tensor(x["input_ids"])
        if not isinstance(x["input_ids"], torch.Tensor)
        else x["input_ids"]
        for x in batch
    ]
    # Applica il padding per portare tutte le cellule alla lunghezza massima del batch
    padded_input_ids = pad_sequence(
        input_ids_list, batch_first=True, padding_value=0
    )
    return {"input_ids": padded_input_ids}


In [34]:
# 3. Crea il DataLoader passando la collate_fn
batch_size = 8
dataloader = DataLoader(ds, batch_size=batch_size, collate_fn=geneformer_collate_fn)

# 4. Estrai il primo batch
batch = next(iter(dataloader))

In [35]:
# 5. Ispeziona le dimensioni
input_ids = batch["input_ids"]

print(f"Tipo di oggetto: {type(input_ids)}")
print(f"Dimensioni del tensore (Batch x Seq_Len): {input_ids.shape}")
print(f"Dimensioni singole:")
print(f"  - Batch size (numero cellule): {input_ids.shape[0]}")
print(f"  - Sequence length (numero token/geni per cellula): {input_ids.shape[1]}")

Tipo di oggetto: <class 'torch.Tensor'>
Dimensioni del tensore (Batch x Seq_Len): torch.Size([8, 4096])
Dimensioni singole:
  - Batch size (numero cellule): 8
  - Sequence length (numero token/geni per cellula): 4096
